# SignalScout - zone classifier (Kaggle-ready, self-contained)

Reproduces the radio zone classifier in one notebook, without the rest of the codebase, so it can run on Kaggle
(**Add data → "4G LTE Speed Dataset" by aeryss**) or locally from `data/raw/lte_speed`.
Training takes about a minute on CPU; SignalScout trains locally by default (`python ml/train_all.py`), so Kaggle is optional.

The label rule and features below mirror `backend/app/ml/signal_ranges.py` and `backend/app/ml/features.py`.

In [1]:
import glob, os
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

candidates = [Path("/kaggle/input/lte-dataset/Dataset"), Path("../data/raw/lte_speed"), Path("data/raw/lte_speed")]
DATA = next(p for p in candidates if p.exists())
files = sorted(glob.glob(str(DATA / "*" / "*.csv")))
df = pd.concat([pd.read_csv(f, na_values=["-"]).assign(trace=os.path.relpath(f, DATA), mobility=Path(f).parent.name) for f in files], ignore_index=True)
df = df[df.Operatorname.astype(str).isin(["A", "B"])]
print(DATA, df.shape)

..\data\raw\lte_speed (174243, 22)


In [2]:
FAM = {"LTE": "lte_nr", "HSPA+": "3g", "HSUPA": "3g", "HSDPA": "3g", "UMTS": "3g", "EDGE": "2g", "GPRS": "2g"}
df["family"] = df.NetworkMode.map(FAM)
df["ts"] = pd.to_datetime(df.Timestamp, format="%Y.%m.%d_%H.%M.%S")
lte = df.family == "lte_nr"
df["level"] = df.RSRP.where(df.RSRP.between(-140, -25)).astype(float)
df["quality"] = df.RSRQ.where(df.RSRQ.between(-34, 3)).astype(float)
df["sinr"] = df.SNR.where(lte & df.SNR.between(-30, 40)).astype(float)
df["cqi"] = df.CQI.where(lte).astype(float)
df["rssi"] = df.RSSI.where(lte).astype(float)
RANGES = {"lte_nr": {"level": (-100, -115), "quality": (-15, None), "sinr": (5, -3)}, "3g": {"level": (-95, -105), "quality": (-14, None)}, "2g": {"level": (-85, -100)}}

def bands(frame):
    out = {}
    for m in ("level", "quality", "sinr"):
        b = pd.Series(np.nan, index=frame.index)
        for fam, rng in RANGES.items():
            if m in rng:
                wb, db = rng[m]; mask = (frame.family == fam) & frame[m].notna(); v = frame.loc[mask, m]
                cls = np.where(v >= wb, 0, 1); cls = np.where(v < db, 2, cls) if db is not None else cls
                b[mask] = cls
        out[f"band_{m}"] = b
    out = pd.DataFrame(out); out["band_worst"] = out.max(axis=1)
    return out

df["label"] = bands(df)["band_worst"]
df = df[df.label.notna()].copy(); df["label"] = df.label.astype(int)
print(df.label.value_counts(normalize=True).rename({0: "Strong", 1: "Weak", 2: "Dead"}).round(3))

label
Strong    0.539
Weak      0.329
Dead      0.132
Name: proportion, dtype: float64


In [3]:
PROFILES = {"full": [], "no_sinr_cqi": ["sinr", "cqi"], "level_only": ["quality", "sinr", "rssi", "cqi"], "rssi_only": ["level", "quality", "sinr", "cqi"]}

def features(frame):
    f = pd.DataFrame(index=frame.index)
    for fam in ("2g", "3g", "lte_nr"):
        f[f"fam_{fam}"] = (frame.family == fam).astype(float)
    for m in ("level", "quality", "sinr", "rssi", "cqi"):
        f[m] = frame[m]; f[f"has_{m}"] = frame[m].notna().astype(float)
    ts = frame.ts.to_numpy()
    for col, stat in (("level", "median"), ("level", "min"), ("level", "std"), ("quality", "median"), ("sinr", "median"), ("rssi", "median")):
        out = np.full(len(frame), np.nan)
        values = frame[col].to_numpy()
        for idx in frame.groupby("device").indices.values():          # 30-second window per device
            idx = idx[np.argsort(ts[idx], kind="stable")]
            w = pd.Series(values[idx], index=pd.DatetimeIndex(ts[idx])).rolling("30s", min_periods=1)
            out[idx] = (w.std(ddof=0) if stat == "std" else getattr(w, stat)()).to_numpy()
        f[f"{col}_{stat}30"] = out
    f["level_dev"] = f.level - f.level_median30
    return pd.concat([f, bands(frame)], axis=1)

def augmented(keys):
    parts = []
    for prof, drop in PROFILES.items():
        sub = df[df.trace.isin(keys)].copy()
        sub[drop] = np.nan
        if prof == "level_only": sub = sub[sub.level.notna()]
        if prof == "rssi_only": sub = sub[sub.rssi.notna()]
        parts.append(sub.assign(device=sub.trace + "|" + prof, profile=prof))
    frame = pd.concat(parts, ignore_index=True)
    return frame, features(frame)

rng = np.random.default_rng(42)
traces = df.groupby("trace").mobility.first()
test_keys = [t for m, grp in traces.groupby(traces) for t in rng.permutation(grp.index)[: max(1, round(0.15 * len(grp)))]]
train_keys = [t for t in traces.index if t not in set(test_keys)]
tr, Xtr = augmented(train_keys); te, Xte = augmented(test_keys)
print("train rows", len(tr), "test rows", len(te))

train rows 550176 test rows 81917


In [4]:
w = tr.label.map(len(tr) / (3 * tr.label.value_counts()))
model = XGBClassifier(n_estimators=300, max_depth=7, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
                      tree_method="hist", objective="multi:softprob", n_jobs=-1, random_state=42)
model.fit(Xtr, tr.label, sample_weight=w)
pred = model.predict(Xte)
print({p: round(f1_score(te.label[te.profile == p], pred[(te.profile == p).to_numpy()], average="macro"), 3) for p in PROFILES})
out = Path("/kaggle/working")
if out.exists():   # on Kaggle: keep the trained model as a notebook output
    model.save_model(str(out / "zone_classifier_xgb.json")); print("saved", out / "zone_classifier_xgb.json")

{'full': 1.0, 'no_sinr_cqi': 0.857, 'level_only': 0.694, 'rssi_only': 0.675}
